# Business Entity Resolution: v2.4

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. Settings: CPU (no accelerator), Internet On. It runs on Kaggle's servers even if you close the tab.

What's new compared with v1:
0. **Better cleaning and a third search:** zero-padded numbers (`004` = `4`), `181ND` = `181st` = `181`, spelling rules that suit transliterated Indian-script names, and a search by name + house number. The shortlist now contains 98.3% of true matches (v1: 97.7%).
1. **Stage 2** looks at all candidates of one business together, from Source 2 and Source 3. The copies of one business look alike, so a candidate that resembles the other strong candidates gets a boost.
2. **Competition:** each Source 2/3 record belongs to at most one business. If a record fits *another* business better, it's probably not ours. To learn this, training uses **every** business of 4 states (Karnataka, Kerala, Ohio, Arizona), 253k in total, the same way the test set has every business.
3. **Empty-address records** (no address, name only) get their own model. It searches all businesses of the country by name and picks the single best owner.

On 236k training businesses that v1 never saw (every business of those 4 states): v1 0.9504, v2.3 0.9674, **v2.4 0.9717**.

`candidate_pairs.tsv` keeps at most 20 candidates per business (about 7 on average), as the organizers now review it.

Cells: 1 install, 2 v1 pipeline code, 3 settings, 4 v3 code, 5 **train** (prints `V2.4 BEST val F0.5`), 6 **predict** the test set and write `output/matching_results.tsv`, 7 official format check.
No external data or APIs are used, only the competition files.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
ORD = {"first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5", "sixth": "6", "seventh": "7",
       "eighth": "8", "ninth": "9", "tenth": "10"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"(\d+)(st|nd|rd|th)\b", r"\1", s)           # 181nd / 181st / 4th -> 181 / 4
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\b0+(\d)", r"\1", s)                         # zero-padded numbers: 0030 -> 30
    s = " ".join(ORD.get(t, t) for t in s.split())             # first -> 1, second -> 2 ...
    return s

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks").replace("c", "k")
    t = t[0] + re.sub(r"[aeiouyh]", "", t[1:])     # also drop h: Kannada/Hindi transliterations add or lose it
    return re.sub(r"(.)\1+", r"\1", t)

LEGAL_RAW = ["limited", "ltd", "private", "pvt", "llc", "llp", "inc", "incorporated", "corp", "corporation", "co",
             "company", "plc", "srl", "sas", "sa", "sc", "sarl", "snc", "gmbh", "dba", "lp", "pc", "pllc", "pa", "ms",
             "and", "the", "c", "t", "m", "s", "d", "l", "p"]
LEGAL_SK = {skel_tok(w) for w in LEGAL_RAW}

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        nums = " ".join(t for t in an.split() if t.isdigit() and len(t) >= 2)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr), "".join(core) + "|" + nums))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states", "combo"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

def _combo_an(s):
    """name letters (3-letter pieces) + house/plot numbers of the address, in one search key"""
    name, _, nm = s.partition("|")
    name = f" {name} "
    return [name[i:i + 3] for i in range(len(name) - 2)] + ["#" + x for x in nm.split()]
def _vec_combo():
    return TfidfVectorizer(analyzer=_combo_an, dtype=np.float32, sublinear_tf=True)

SEARCH = (("name_skc", _vec_name), ("addr_sk", _vec_addr), ("combo", _vec_combo))

def _rowdot(X, Y, r, c):
    """cosine similarity of X[r[i]] and Y[c[i]] for each i (rows are already unit length)"""
    return np.asarray(X[r].multiply(Y[c]).sum(axis=1)).ravel().astype(np.float32)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3).
    Three searches: by name, by address, and by name + house number."""
    def __init__(self, B, k_state=15, k_unk=10, k_combo=10):
        self.B, self.k_state, self.k_unk, self.k_combo = B, k_state, k_unk, k_combo
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in SEARCH:
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos]).tocsr()
                entry[col] = (v, M, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr, found); a = row in A, b = row in self.B.
        sim_name / sim_addr are computed for EVERY shortlisted pair; found = which searches found it (bits 1/2/4)."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for i in range(0, len(rows), chunk):
                rr = rows[i:i + chunk]; Q = {}; R, Cc, F = [], [], []
                for bit, (col, _) in zip((1, 2, 4), SEARCH):
                    if e[col] is None:
                        continue
                    v, M, BT = e[col]
                    Q[col] = v.transform(A[col].to_numpy()[rr])
                    kk = self.k_combo if col == "combo" else k
                    C = sp_matmul_topn(Q[col], BT, top_n=kk, threshold=thr if col != "combo" else 0.05,
                                       n_threads=N_CPU).tocoo()
                    R.append(C.row); Cc.append(C.col); F.append(np.full(len(C.row), bit, np.int8))
                if not R:
                    continue
                pr = pd.DataFrame({"r": np.concatenate(R), "c": np.concatenate(Cc), "f": np.concatenate(F)})
                pr = pr.groupby(["r", "c"], sort=False).f.sum().reset_index()
                r, c = pr.r.to_numpy(), pr.c.to_numpy()
                sims = {}
                for col in ("name_skc", "addr_sk"):
                    sims[col] = _rowdot(Q[col], e[col][1], r, c) if col in Q else np.full(len(r), np.nan, np.float32)
                out.append((rr[r], e["pos"][c], sims["name_skc"], sims["addr_sk"], pr.f.to_numpy()))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": [], "found": []})
        df = pd.DataFrame({"key": np.concatenate([o[0] for o in out]).astype(np.int64) * len(self.B)
                                  + np.concatenate([o[1] for o in out]).astype(np.int64),
                           "sim_name": np.concatenate([o[2] for o in out]),
                           "sim_addr": np.concatenate([o[3] for o in out]),
                           "found": np.concatenate([o[4] for o in out])})
        df = df.groupby("key", sort=False).agg(sim_name=("sim_name", "max"), sim_addr=("sim_addr", "max"),
                                               found=("found", "max")).reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    if "found" in P: f["found"] = P.found.to_numpy().astype(np.int8)
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # models + outputs are saved here
REGION = ["ka", "kl", "oh", "az"]   # train on EVERY business of these states (Karnataka, Kerala, Ohio, Arizona):
                                    # like the test set, all competing businesses are present
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)
KEEP_P = 0.01        # stage-1 filter: candidates below this chance are dropped before stage 2
K_CTX = 8            # stage 2 compares each candidate with this many of the business's best candidates
KE = 10              # empty-address records: how many Source 1 look-alikes (by name) to consider
MAX_CANDS = 20       # candidate_pairs.tsv keeps at most this many candidates per business
SMOKE = 0             # 1 = tiny quick test run (for checking the code only)
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU, "| smoke test:", bool(SMOKE))


In [ ]:
# =====================================================================
# Version 2.4 = v1 pipeline (with better cleaning and a 3rd search: name + house number) + three additions:
#  (a) STAGE 2 looks at all candidates of a business together. One business shows up ~3 times across Source 2/3
#      and the copies look alike, so a candidate that resembles the other strong candidates gets a boost.
#  (b) COMPETITION: each Source 2/3 record belongs to at most one business. If a record fits ANOTHER business
#      better, it is probably not ours. (Trained on every business of a few states, so competitors are present.)
#  (c) EMPTY-ADDRESS records can only be matched by name. They get their own model that searches ALL Source 1
#      businesses of the country by name and picks the best owner.
# =====================================================================
from sklearn.model_selection import GroupKFold

def load_src(name):
    """Load + clean one Source 2/3 file, saved to disk the first time (later calls read it back).
    Cleaning uses several worker processes; doing it early, while memory is still empty, avoids out-of-memory."""
    path = f"{WORK}/prep_{name}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    B = load(ROOT, name)
    if SMOKE:
        B = B.sample(min(len(B), 1_000_000), random_state=0).reset_index(drop=True)
    B = prepare(B); B.to_parquet(path)
    return B

def s1_num(ids):
    """'S1-123' -> 123 (numbers use far less memory than text)"""
    return pd.Series(ids).str.slice(3).astype(np.int64).to_numpy()
def cand_num(ids):
    """'S2-123' -> 123*4+2, 'S3-123' -> 123*4+3 (unique number per Source 2/3 record)"""
    s = pd.Series(ids)
    return (s.str.slice(3).astype(np.int64) * 4 + 2 + (s.str[1] == "3")).to_numpy()
def cand_str(n):
    return np.array([f"S{2 + (x & 1)}-{x >> 2}" for x in n.tolist()], dtype=object)

# ---------- main pairs: shortlist + stage-1 features, written to disk batch by batch ----------
def write_pairs(A, B, is_s3, src, outdir, booster=None, feats=None):
    """For every S1 in A: shortlist within B (same country/state), features, save a parquet per batch.
    With a booster (test time) only candidates with stage-1 chance >= KEEP_P are kept."""
    for country in A.country.unique():
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        Bc = B[B.country == country].reset_index(drop=True)
        if len(Bc) == 0: continue
        todo = [i for i in range(0, len(a_rows), BATCH) if not os.path.exists(f"{outdir}/{src}_{country}_{i}.parquet")]
        if not todo: continue
        idx = Index(Bc)
        for i in todo:
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = idx.query(Ac)
            F = features(P, Ac, Bc, is_s3).astype(np.float32) if len(P) else pd.DataFrame()
            b = P.b.to_numpy()
            F["s1"] = s1_num(Ac.entity_id.to_numpy()[P.a.to_numpy()]); F["cand"] = cand_num(Bc.entity_id.to_numpy()[b])
            F["cname"] = Bc.name_sk.to_numpy()[b]; F["caddr"] = Bc.addr_sk.to_numpy()[b]
            F["noaddr"] = (Bc.addr_n.to_numpy()[b] == "").astype(np.int8); F["s3"] = np.int8(is_s3)
            if booster is not None:
                F["p1"] = booster.predict(F[feats].to_numpy(np.float32), num_threads=N_CPU).astype(np.float32) if len(F) else np.zeros(0, np.float32)
                F = F[F.p1 >= KEEP_P]
            F.reset_index(drop=True).to_parquet(f"{outdir}/{src}_{country}_{i}.parquet")
            log(src, country, f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}", "shortlist", len(P), "saved", len(F))
            del P, F; gc.collect()
        del idx; gc.collect()

# ---------- stage 2 features ----------
S2_EXTRA = ["p1", "p1_rank", "p1_max", "p1_2nd", "p1_sum", "p1_rel", "n_hi", "n_kept",
            "sup_max", "sup_sum", "sup_cnt", "sup_name_max", "sup_addr_max", "sup_other_src"]

def stage2_features(key, p1, cname, caddr, is_s3, K=K_CTX):
    """Context features. key: business of each row; p1: stage-1 chance; cname/caddr: candidate's cleaned text."""
    d = pd.DataFrame({"key": key, "p1": p1.astype(np.float32), "row": np.arange(len(key)), "s3": is_s3})
    d = d.sort_values(["key", "p1"], ascending=[True, False], kind="stable")
    g = d.groupby("key", sort=False)
    d["p1_rank"] = (g.cumcount() + 1).astype(np.float32)
    d["p1_max"] = g.p1.transform("max")
    d["p1_2nd"] = d.key.map(d[d.p1_rank == 2].set_index("key").p1).fillna(0).to_numpy(np.float32)
    d["p1_sum"] = g.p1.transform("sum")
    d["p1_rel"] = d.p1 / d.p1_max.clip(lower=1e-6)
    d["n_hi"] = (d.p1 > 0.5).groupby(d.key).transform("sum").astype(np.float32)
    d["n_kept"] = g.p1.transform("size").astype(np.float32)
    top = d[d.p1_rank <= K][["key", "row", "p1", "s3"]]
    j = top.merge(top, on="key", suffixes=("", "_o"))
    j = j[j.row != j.row_o]
    r, ro = j.row.to_numpy(), j.row_o.to_numpy()
    sn = cpdist(cname[r].tolist(), cname[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    sa = cpdist(caddr[r].tolist(), caddr[ro].tolist(), scorer=fuzz.token_set_ratio, workers=N_CPU).astype(np.float32) / 100
    po = j.p1_o.to_numpy()
    s = (sn + sa) / 2
    j = pd.DataFrame({"row": r, "w": po * s, "strong": ((s > 0.85) & (po > 0.5)).astype(np.int8),
                      "wn": po * sn, "wa": po * sa, "wx": np.where(j.s3.to_numpy() != j.s3_o.to_numpy(), po * s, 0)})
    agg = j.groupby("row").agg(sup_max=("w", "max"), sup_sum=("w", "sum"), sup_cnt=("strong", "sum"),
                               sup_name_max=("wn", "max"), sup_addr_max=("wa", "max"), sup_other_src=("wx", "max"))
    d = d.set_index("row").sort_index().join(agg)
    return d[S2_EXTRA].fillna(0).astype(np.float32).reset_index(drop=True)

def stage2_in_chunks(key, *cols, chunk=200_000):
    """stage2_features for many businesses, a block of businesses at a time to save memory."""
    order = np.argsort(key, kind="stable"); ks = key[order]
    uk = np.unique(ks); out = []
    for i in range(0, len(uk), chunk):
        lo, hi = np.searchsorted(ks, uk[i]), np.searchsorted(ks, uk[min(i + chunk, len(uk)) - 1], side="right")
        sel = order[lo:hi]
        f = stage2_features(key[sel], *[c[sel] for c in cols]); f.index = sel; out.append(f)
    return pd.concat(out).sort_index().reset_index(drop=True)

def comp_features(cand, p1):
    """Competition: how strongly does this record fit OTHER businesses?"""
    c = pd.DataFrame({"cand": cand, "p1": p1})
    g = c.groupby("cand").p1
    mx = g.transform("max").to_numpy()
    r = g.rank(ascending=False, method="first")
    second = c.cand.map(c[r == 2].set_index("cand").p1).fillna(0).to_numpy()
    other = np.where(p1 >= mx, second, mx)
    return pd.DataFrame({"c_n": g.transform("size").to_numpy(np.float32), "c_other_max": other.astype(np.float32),
                         "c_margin": (p1 - other).astype(np.float32),
                         "c_rank": g.rank(ascending=False, method="min").to_numpy(np.float32)})

# ---------- empty-address records: name-only matching against ALL Source 1 of the country ----------
E_DROP = ("sim_addr", "ask_tset", "ask_ratio", "ask_partial", "num_jacc", "num_contain", "state_rel", "cand_addr_len",
          "rank_s1", "gap_s1", "n_cand_s1")
def emod_pairs(A, E, is_s3):
    out = []
    for country in E.country.unique():
        Ac = A[A.country == country].reset_index(drop=True); Ec = E[E.country == country].reset_index(drop=True)
        if not len(Ac) or not len(Ec): continue
        v = TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
        MA = v.fit_transform(Ac.name_skc.to_numpy()).T.tocsr()
        Cm = sp_matmul_topn(v.transform(Ec.name_skc.to_numpy()), MA, top_n=KE, threshold=0.1, n_threads=N_CPU).tocoo()
        P = pd.DataFrame({"a": Cm.col.astype(np.int64), "b": Cm.row.astype(np.int64), "sim_name": Cm.data, "sim_addr": np.nan})
        F = features(P, Ac, Ec, is_s3).drop(columns=list(E_DROP), errors="ignore")
        F["s1"] = s1_num(Ac.entity_id.to_numpy()[P.a]); F["cand"] = cand_num(Ec.entity_id.to_numpy()[P.b])
        F["s1_skc"] = Ac.name_skc.to_numpy()[P.a]
        out.append(F); log("empty-address", country, "S3" if is_s3 else "S2", "records", len(Ec), "pairs", len(P))
    return pd.concat(out, ignore_index=True) if out else None

def emod_context(F, namefreq):
    v = F.nb_tset.to_numpy() + F.nsk_jw.to_numpy() * 100 + F.sim_name.to_numpy() * 100
    t = pd.DataFrame({"c": F.cand.to_numpy(), "v": v}); g = t.groupby("c")["v"]
    F["r_rank"] = g.rank(ascending=False, method="min").to_numpy(np.float32)
    F["r_gap"] = (g.transform("max").to_numpy() - v).astype(np.float32)
    sec = t.sort_values(["c", "v"], ascending=[True, False]); sec["r"] = sec.groupby("c").cumcount()
    F["r_margin"] = (v - F.cand.map(sec[sec.r == 1].set_index("c").v).fillna(0).to_numpy()).astype(np.float32)
    F["r_n"] = g.transform("size").to_numpy(np.float32)
    F["r_n_exact"] = F.groupby("cand").name_exact.transform("sum").astype(np.float32)
    gs = pd.DataFrame({"s": F.s1.to_numpy(), "v": v}).groupby("s")["v"]
    F["f_n"] = gs.transform("size").to_numpy(np.float32)
    F["f_rank"] = gs.rank(ascending=False, method="min").to_numpy(np.float32)
    F["s1_name_freq"] = F.s1_skc.map(namefreq).fillna(1).to_numpy(np.float32)
    return F.drop(columns="s1_skc")

def name_freq(A):
    f = A.groupby("name_skc").size()
    return f[f > 1]

def decide3(main, emp, tm, te):
    """main: DataFrame(s1, cand, p) for records WITH an address; emp: same for empty-address records.
    Each record goes to at most one business (its best), and only if its chance passes the cut-off."""
    d = main[main.p >= tm].sort_values("p", ascending=False).drop_duplicates("cand")
    e = emp.sort_values("p", ascending=False).drop_duplicates("cand"); e = e[e.p >= te]
    out = {}
    for s, c in zip(np.concatenate([d.s1.to_numpy(), e.s1.to_numpy()]), np.concatenate([d.cand.to_numpy(), e.cand.to_numpy()])):
        out.setdefault(s, set()).add(c)
    return out


In [ ]:
# ================= TRAINING =================
for src in ("train_source2", "train_source3"):          # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading train Source 1 + ground truth")
A_all = prepare(load(ROOT, "train_source1")); gt = load(ROOT, "train_ground_truth")
owner = {}                                   # Source 2/3 record number -> its true business number
truth = {}                                   # business number -> set of its true record numbers
for s, m in zip(s1_num(gt.source1_entity_id.to_numpy()), gt.matched_entity_ids.to_numpy()):
    cs = [c for c in m.split(",") if c]
    ns = set(cand_num(cs).tolist()) if cs else set()
    truth[s] = ns
    for c in ns: owner[c] = s
del gt; gc.collect()
want = set(REGION)
A = A_all[A_all.states.map(lambda s: bool(set(s.split("|")) & want))].reset_index(drop=True)
if SMOKE: A = A.sample(20_000, random_state=0).reset_index(drop=True)
log("training businesses (all of", REGION, "):", len(A), A.country.value_counts().to_dict())

# ---------- 1. shortlist + stage-1 features ----------
TP = f"{WORK}/train_pairs"; os.makedirs(TP, exist_ok=True)
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    B = load_src(src); write_pairs(A, B, is_s3, src, TP); del B; gc.collect()
files = sorted(glob.glob(f"{TP}/*.parquet"))
TXT = ("s1", "cand", "cname", "caddr", "noaddr", "s3")
FEAT1 = [c for c in pd.read_parquet(files[0]).columns if c not in TXT]
Xs, S, Cd, L = [], [], [], []
for f in files:                               # load numbers only (text columns are read later, for fewer rows)
    t = pd.read_parquet(f, columns=FEAT1 + ["s1", "cand"])
    Xs.append(t[FEAT1]); S.append(t.s1.to_numpy()); Cd.append(t.cand.to_numpy()); L.append(len(t)); del t
X = pd.concat(Xs, ignore_index=True); s1n = np.concatenate(S); cdn = np.concatenate(Cd); del Xs, S, Cd; gc.collect()
y = np.fromiter((owner.get(c, -1) == s for s, c in zip(s1n.tolist(), cdn.tolist())), bool, len(s1n)).astype(np.int8)
ids = s1_num(A.entity_id.to_numpy()); vt = {s: truth.get(s, set()) for s in ids.tolist()}; vl = ids.tolist()
all_true = sum(len(v) for v in vt.values())
log("pairs", len(X), "per S1", round(len(X) / len(A), 1), "| shortlist recall", round(y.sum() / all_true, 4))

# ---------- 2. stage 1: out-of-fold chances (each business scored by a model that never saw it) ----------
P1 = dict(n_estimators=400, learning_rate=0.08, num_leaves=255, min_child_samples=100, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
if SMOKE: P1["n_estimators"] = 60
p1 = np.zeros(len(X), np.float32)
for k, (tr, te) in enumerate(GroupKFold(n_splits=3).split(X, y, groups=s1n)):
    m = lgb.LGBMClassifier(**P1).fit(X.iloc[tr], y[tr]); p1[te] = m.predict_proba(X.iloc[te])[:, 1]
    log("stage-1 fold", k + 1, "of 3 done"); del m; gc.collect()
m = lgb.LGBMClassifier(**P1).fit(X, y); m.booster_.save_model(f"{WORK}/v24_stage1.txt"); del m; gc.collect()
log("stage-1 only (like v1), val F0.5 at 0.65:",
    round(macro_f05(decide(pd.DataFrame({"s1_id": s1n, "cand_id": cdn, "p": p1}), 0.65), vt, vl), 4))

# ---------- 3. keep candidates with a real chance; stage-2 + competition features ----------
keep = p1 >= KEEP_P
log("kept after stage 1:", int(keep.sum()), "| true matches kept", round(y[keep].sum() / all_true, 4))
X = X[keep].reset_index(drop=True); y = y[keep]; p1 = p1[keep]; s1n = s1n[keep]; cdn = cdn[keep]
cn, ca, em, s3 = [], [], [], []; o = 0
for f, l in zip(files, L):
    k = keep[o:o + l]; o += l
    t = pd.read_parquet(f, columns=["cname", "caddr", "noaddr", "s3"])[k]
    cn.append(t.cname.to_numpy()); ca.append(t.caddr.to_numpy()); em.append(t.noaddr.to_numpy()); s3.append(t.s3.to_numpy())
cn, ca, em, s3 = map(np.concatenate, (cn, ca, em, s3)); gc.collect()
X2 = pd.concat([X, stage2_in_chunks(s1n, p1, cn, ca, s3), comp_features(cdn, p1)], axis=1); del X, cn, ca; gc.collect()
FEAT2 = list(X2.columns); log("stage-2 features", X2.shape)
P2 = dict(n_estimators=600, learning_rate=0.05, num_leaves=255, min_child_samples=100, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
if SMOKE: P2["n_estimators"] = 60
p2 = np.zeros(len(X2), np.float32)
for k, (tr, te) in enumerate(GroupKFold(n_splits=3).split(X2, y, groups=s1n)):
    m = lgb.LGBMClassifier(**P2).fit(X2.iloc[tr], y[tr]); p2[te] = m.predict_proba(X2.iloc[te])[:, 1]
    log("stage-2 fold", k + 1, "of 3 done"); del m; gc.collect()
m = lgb.LGBMClassifier(**P2).fit(X2, y); m.booster_.save_model(f"{WORK}/v24_stage2.txt")
log("top stage-2 features:\n" + pd.Series(m.feature_importances_, FEAT2).sort_values(ascending=False).head(10).to_string())
del m, X2; gc.collect()
main = pd.DataFrame({"s1": s1n, "cand": cdn, "p": p2, "noaddr": em})
log("stage 2 (all records), val F0.5 at 0.7:", round(macro_f05(decide(main.rename(columns={"s1": "s1_id", "cand": "cand_id"}), 0.7), vt, vl), 4))

# ---------- 4. empty-address model (trained on ALL training businesses) ----------
NF = name_freq(A_all); Es = []
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    B = load_src(src); E = B[B.addr_n == ""].reset_index(drop=True); del B; gc.collect()
    if SMOKE: E = E.head(20_000)
    Es.append(emod_pairs(A_all, E, is_s3)); del E; gc.collect()
FE_ = emod_context(pd.concat([e for e in Es if e is not None], ignore_index=True), NF); del Es
ye = np.fromiter((owner.get(c, -1) == s for s, c in zip(FE_.s1.tolist(), FE_.cand.tolist())), bool, len(FE_)).astype(np.int8)
FEATE = [c for c in FE_.columns if c not in ("s1", "cand")]
PE = dict(n_estimators=500, learning_rate=0.05, num_leaves=127, min_child_samples=50, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
pe = np.zeros(len(FE_), np.float32)
for k, (tr, te) in enumerate(GroupKFold(n_splits=3).split(FE_, ye, groups=FE_.s1)):
    m = lgb.LGBMClassifier(**PE).fit(FE_[FEATE].iloc[tr], ye[tr]); pe[te] = m.predict_proba(FE_[FEATE].iloc[te])[:, 1]
m = lgb.LGBMClassifier(**PE).fit(FE_[FEATE], ye); m.booster_.save_model(f"{WORK}/v24_empty.txt"); del m
emp = pd.DataFrame({"s1": FE_.s1.to_numpy(), "cand": FE_.cand.to_numpy(), "p": pe}); del FE_; gc.collect()
emp_best = emp.sort_values("p", ascending=False).drop_duplicates("cand")      # best owner among ALL businesses
emp_best = emp_best[emp_best.s1.isin(set(vl))]

# ---------- 5. pick the two cut-offs on the training states ----------
best = (0, None)
for tm in (0.5, 0.6, 0.65, 0.7, 0.75, 0.8):
    for te in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
        s = macro_f05(decide3(main[main.noaddr == 0], emp_best, tm, te), vt, vl)
        if s > best[0]: best = (s, (tm, te))
log("V2.4 BEST val F0.5", round(best[0], 4), "at cut-offs (main, empty-address)", best[1], "| stage 1 alone: see line above")
json.dump({"tm": best[1][0], "te": best[1][1], "val_f05": best[0], "features1": FEAT1, "features2": FEAT2, "featuresE": FEATE},
          open(f"{WORK}/v24_meta.json", "w"))
del main, emp, emp_best, y, p1, p2, A, A_all, owner, truth; gc.collect()
import shutil; shutil.rmtree(TP, ignore_errors=True)
for src in ("train_source2", "train_source3"):
    os.remove(f"{WORK}/prep_{src}.parquet")              # free disk space


In [ ]:
# ================= PREDICTION ON THE TEST SET =================
meta = json.load(open(f"{WORK}/v24_meta.json"))
b1 = lgb.Booster(model_file=f"{WORK}/v24_stage1.txt"); b2 = lgb.Booster(model_file=f"{WORK}/v24_stage2.txt")
bE = lgb.Booster(model_file=f"{WORK}/v24_empty.txt")
for src in ("test_source2", "test_source3"):            # clean both sources first (saved to disk)
    log("cleaning", src); B = load_src(src); log("cleaned", src, B.shape); del B; gc.collect()
log("loading test Source 1"); T = load(ROOT, "test_source1")
if SMOKE: T = T.groupby("country").head(10_000).reset_index(drop=True)
A = prepare(T); del T; log("prepared", A.shape); print(A.country.value_counts().to_string())
s1_ids = A.entity_id.to_numpy()

# ---------- stage 1 per batch (resumable: finished batches are skipped) + empty-address model ----------
PP = f"{WORK}/parts_v24"; os.makedirs(PP, exist_ok=True)
NF = name_freq(A)
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    B = load_src(src)
    write_pairs(A, B, is_s3, src, PP, booster=b1, feats=meta["features1"])
    out = f"{PP}/empty_{src}.parquet"
    if not os.path.exists(out):
        E = B[B.addr_n == ""].reset_index(drop=True)
        FE_ = emod_pairs(A, E, is_s3)
        if FE_ is not None:
            FE_ = emod_context(FE_, NF)
            pd.DataFrame({"s1": FE_.s1.to_numpy(), "cand": FE_.cand.to_numpy(),
                          "p": bE.predict(FE_[meta["featuresE"]].to_numpy(np.float32), num_threads=N_CPU)}).to_parquet(out)
        else:
            pd.DataFrame({"s1": np.zeros(0, np.int64), "cand": np.zeros(0, np.int64), "p": np.zeros(0)}).to_parquet(out)
        del E, FE_; gc.collect()
    del B; gc.collect()

# ---------- stage 2 on all candidates of both sources together ----------
log("stage 2")
parts = [f for f in sorted(glob.glob(f"{PP}/*.parquet")) if not os.path.basename(f).startswith("empty_")]
M = pd.concat([pd.read_parquet(f) for f in parts], ignore_index=True)
M = M[M.cand.notna()].reset_index(drop=True)
s1n = M.s1.to_numpy().astype(np.int64); cdn = M.cand.to_numpy().astype(np.int64); p1 = M.p1.to_numpy(np.float32)
X2 = pd.concat([M[meta["features1"]].reset_index(drop=True),
                stage2_in_chunks(s1n, p1, M.cname.to_numpy(), M.caddr.to_numpy(), M.s3.to_numpy()),
                comp_features(cdn, p1)], axis=1)[meta["features2"]]
main = pd.DataFrame({"s1": s1n, "cand": cdn, "p": b2.predict(X2.to_numpy(np.float32), num_threads=N_CPU),
                     "noaddr": M.noaddr.to_numpy()})
del X2, M; gc.collect()
emp = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{PP}/empty_*.parquet"))], ignore_index=True)
match = decide3(main[main.noaddr == 0], emp, meta["tm"], meta["te"])

# ---------- write the two output files ----------
num2id = dict(zip(s1_num(s1_ids).tolist(), s1_ids))
match = {num2id[s]: cand_str(np.fromiter(c, np.int64)) for s, c in match.items()}
# candidate_pairs.tsv: at most MAX_CANDS candidates per business (best first), always including the final matches
e_top = emp.sort_values("p", ascending=False).groupby("s1").head(5)
cands = pd.concat([main[["s1", "cand", "p"]], e_top[["s1", "cand", "p"]]]).sort_values("p", ascending=False)
cands = cands.drop_duplicates(["s1", "cand"]).groupby("s1").head(MAX_CANDS)
cmap = {num2id[s]: set(cand_str(g.to_numpy())) for s, g in cands.groupby("s1").cand}
for s, m in match.items():
    cmap.setdefault(s, set()).update(m)
log("candidate pairs per business:", round(sum(len(v) for v in cmap.values()) / len(s1_ids), 2))
del main, emp, cands; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
write_tsv(f"{WORK}/output/matching_results.tsv", s1_ids, match, "matched_entity_ids")
write_tsv(f"{WORK}/output/candidate_pairs.tsv", s1_ids, cmap, "candidate_entity_ids")
n = sum(1 for s in s1_ids if len(match.get(s, ())))
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(sum(len(v) for v in match.values()) / len(s1_ids), 2))
for src in ("test_source2", "test_source3"):
    os.remove(f"{WORK}/prep_{src}.parquet")              # free disk space (cleaned copies no longer needed)


In [ ]:
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip. You can delete the `parts_v24/` folder.